# Privacy Lens · 일상 사진 개인정보 (공개 데이터 5개)

교수님 피드백(11-04, D-19): 일부러 넣은 개인정보가 아니라 **일상 사진에 자연스럽게 찍힌 개인정보**를 잡아야 한다.
위치 라벨이 있고 바로 받을 수 있는 공개 데이터만 쓴다(`vision/daily_data.py`). 사진은 **드라이브·런타임에만**, 비영리(수업) 사용.

| 데이터 | 사진 | 라이선스 |
|---|---|---|
| VISPR-redactions | Flickr 일상 사진 8,473장, 24종 | 라벨 CC BY-NC 4.0, 사진 원 라이선스 |
| BIV-Priv-HIT | 개인정보 물건을 배경·전경에 둔 영상 프레임(물체 16종만 씀) | CC BY 4.0 |
| BIV-Priv-Seg | 시각장애인 촬영 1,056장, 16종 | CC BY 4.0 |
| VizWiz-Priv | 시각장애인 촬영, 개인정보는 가짜 내용으로 채움 | CC BY 4.0 |
| Open Images V7 | 모니터·휴대폰·노트북·TV·태블릿 | CC BY 4.0 |

클래스 9종: 신분증 · 결제 카드 · 명함·명찰 · 우편물·송장 · 영수증·고지서 · 개인 서류 · 티켓 · 화면 · 약·건강 관련 물건

| 칸 | 하는 일 | GPU | 시간 |
|---|---|---|---|
| 0 | 드라이브·코드·설치(PaddleOCR 포함) | - | 5분 |
| 1 | **지금 서비스 평가**: VISPR-redactions test 300장, 종류별로 찾은 비율 → `objects/eval_daily_service_300.json` | 필요 없음 | 40~60분 |
| 2 | **데이터 합치기** → 드라이브 `objects/daily_ds.zip` (받기 20GB 안팎) | 필요 없음 | 1~2시간 |
| 3 | **학습**(YOLO26n, 9종, 30 epoch) → `objects/daily_v001/` | 필요 | 2~3시간 |

- 1·2번은 CPU 런타임에서, 3번은 GPU 런타임에서 돌린다(2번 결과가 드라이브 zip 이라 런타임을 바꿔도 된다).
- 학습 기록은 `objects/runs/daily_objects/` 에 저장되어 끊겨도 3번 칸만 다시 돌리면 이어서 한다.
- 해외 사진이라 한국 사진 성능은 팀 실사로 따로 잰다.


In [ ]:
BRANCH = "main"   # 고친 브랜치를 시험할 때만 바꾼다
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
if not os.path.isdir("/content/privacy-lens"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/2026-usw-project/privacy-lens", "/content/privacy-lens"], check=True)
%cd /content/privacy-lens
!GIT_TERMINAL_PROMPT=0 git pull -q --ff-only && git log --oneline -1
!pip -q install -r requirements.txt "ultralytics>=8.4" onnxruntime fiftyone paddleocr "paddlepaddle<3.3" pycocotools
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(CPU)")

In [ ]:
import importlib, vision.next_steps as ns; importlib.reload(ns)
res = ns.eval_daily(limit=300)

In [ ]:
import importlib, vision.next_steps as ns; importlib.reload(ns)
counts = ns.build_daily()

In [ ]:
import importlib, vision.next_steps as ns; importlib.reload(ns)
daily = ns.train_daily(epochs=30)